# 02 — Titanic: leak-free modeling

Build a binary survival classifier and a separate fare-regression baseline. Each workflow splits the data before fitting imputers, encoders, scalers, or estimators; all learned preprocessing lives inside scikit-learn pipelines.

The target is `survived`. Predictors are limited to passenger/travel fields (`pclass`, `sex`, `age`, `sibsp`, `parch`, `fare`, `embarked`). In particular, fields such as `alive` (a duplicate of the target), `who`, and `adult_male` are excluded. The regression task predicts `fare` and excludes `survived` to avoid target leakage.

Look for a local CSV in the working directory, `data/`, or `analytics/`; otherwise the notebook downloads the standard seaborn Titanic CSV from GitHub (internet required on first use).

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42


In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
LOCAL_DATA_CANDIDATES = [
    Path("titanic.csv"),
    Path("data/titanic.csv"),
    Path("analytics/titanic.csv"),
    Path("analytics/data/titanic.csv"),
]

local_data = next((path for path in LOCAL_DATA_CANDIDATES if path.is_file()), None)
if local_data is None:
    titanic = pd.read_csv(DATA_URL)
    DATA_SOURCE = DATA_URL
else:
    titanic = pd.read_csv(local_data)
    DATA_SOURCE = str(local_data.resolve())

titanic.columns = titanic.columns.str.lower()
required_columns = {
    "survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"
}
missing_columns = required_columns.difference(titanic.columns)
if missing_columns:
    raise ValueError(f"Titanic data is missing required columns: {sorted(missing_columns)}")

print(f"Loaded {len(titanic):,} rows from {DATA_SOURCE}")


## 1. Target, predictors, and stratified holdout

A stratified holdout preserves the class balance in both partitions. The test set remains untouched during model selection and tuning.

In [ ]:
FEATURES = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
TARGET = "survived"

model_data = titanic.dropna(subset=[TARGET]).copy()
model_data[TARGET] = pd.to_numeric(model_data[TARGET], errors="raise").astype(int)
if not set(model_data[TARGET].unique()).issubset({0, 1}):
    raise ValueError("Expected the survived target to contain only 0 and 1.")

X = model_data[FEATURES].copy()
y = model_data[TARGET]
NUMERIC_FEATURES = ["pclass", "age", "sibsp", "parch", "fare"]
CATEGORICAL_FEATURES = ["sex", "embarked"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"Training rows: {len(X_train)} | Holdout rows: {len(X_test)}")
display(
    pd.DataFrame(
        {
            "train_fraction_survived": [y_train.mean()],
            "test_fraction_survived": [y_test.mean()],
        },
        index=["survived"],
    ).round(3)
)


## 2. Reusable preprocessing and held-out evaluation

Median numeric imputation, standardization, most-frequent categorical imputation, and one-hot encoding are fitted inside each pipeline. This prevents validation and test rows from influencing preprocessing.

In [ ]:
def make_preprocessor():
    numeric = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric, NUMERIC_FEATURES),
            ("categorical", categorical, CATEGORICAL_FEATURES),
        ],
        remainder="drop",
    )


def classification_metrics(estimator, features, labels):
    prediction = estimator.predict(features)
    probabilities = estimator.predict_proba(features)
    positive_column = list(estimator.classes_).index(1)
    return {
        "accuracy": accuracy_score(labels, prediction),
        "precision": precision_score(labels, prediction, zero_division=0),
        "recall": recall_score(labels, prediction, zero_division=0),
        "f1": f1_score(labels, prediction, zero_division=0),
        "roc_auc": roc_auc_score(labels, probabilities[:, positive_column]),
    }


def make_classifier_pipeline(classifier):
    return Pipeline(
        [
            ("preprocessor", make_preprocessor()),
            ("classifier", classifier),
        ]
    )


## 3. Baseline and three classifiers

The dummy baseline predicts the training-set majority class. The three substantive models are logistic regression, a single decision tree, and a random forest. Every score below is calculated on the same held-out test partition.

In [ ]:
classifiers = {
    "Logistic regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Decision tree": DecisionTreeClassifier(
        min_samples_leaf=5, random_state=RANDOM_STATE
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=1
    ),
}

baseline = make_classifier_pipeline(DummyClassifier(strategy="most_frequent"))
baseline.fit(X_train, y_train)
fitted_models = {"Majority-class baseline": baseline}
for name, classifier in classifiers.items():
    pipeline = make_classifier_pipeline(classifier)
    pipeline.fit(X_train, y_train)
    fitted_models[name] = pipeline

comparison = pd.DataFrame(
    {
        name: classification_metrics(model, X_test, y_test)
        for name, model in fitted_models.items()
    }
).T.sort_values("roc_auc", ascending=False)
display(comparison.style.format("{:.3f}"))

for name in classifiers:
    print(f"\n{name}: held-out classification report")
    print(classification_report(
        y_test,
        fitted_models[name].predict(X_test),
        target_names=["did not survive", "survived"],
        zero_division=0,
    ))
    display(
        pd.DataFrame(
            confusion_matrix(y_test, fitted_models[name].predict(X_test)),
            index=["actual 0", "actual 1"],
            columns=["predicted 0", "predicted 1"],
        )
    )


## 4. Random-forest hyperparameter tuning with OOB scores

`RandomizedSearchCV` selects parameters by mean out-of-bag (OOB) accuracy across stratified folds of the **training partition only**. In each fold the pipeline fits its preprocessing on that fold's training rows, and the bootstrap forest estimates OOB accuracy from samples excluded from each tree. The test set remains untouched for final evaluation. OOB accuracy is a training-only selection signal, not a substitute for the held-out test metrics.

In [ ]:
tuning_pipeline = make_classifier_pipeline(
    RandomForestClassifier(
        bootstrap=True,
        oob_score=True,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
)
parameter_distributions = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [None, 5, 10, 15],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", 0.7],
}
stratified_cv = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=RANDOM_STATE
)
def oob_accuracy_scorer(estimator, features, labels):
    del features, labels
    return estimator.named_steps["classifier"].oob_score_

search = RandomizedSearchCV(
    estimator=tuning_pipeline,
    param_distributions=parameter_distributions,
    n_iter=10,
    scoring=oob_accuracy_scorer,
    cv=stratified_cv,
    refit=True,
    random_state=RANDOM_STATE,
    n_jobs=1,
    return_train_score=True,
)
search.fit(X_train, y_train)

best_pipeline = search.best_estimator_
best_forest = best_pipeline.named_steps["classifier"]
print(f"Best mean CV OOB accuracy: {search.best_score_:.3f}")
print(f"Best parameters: {search.best_params_}")
print(f"Refit training-set OOB score: {best_forest.oob_score_:.3f}")
print("Tuned model held-out metrics:")
display(pd.Series(classification_metrics(best_pipeline, X_test, y_test)).round(3))
print("Tuned model held-out classification report:")
print(classification_report(
    y_test,
    best_pipeline.predict(X_test),
    target_names=["did not survive", "survived"],
    zero_division=0,
))
display(
    pd.DataFrame(
        confusion_matrix(y_test, best_pipeline.predict(X_test)),
        index=["actual 0", "actual 1"],
        columns=["predicted 0", "predicted 1"],
    )
)


## 5. Regression side task: predict fare

This separate task predicts the numeric `fare` value. `survived` is explicitly excluded, and rows without a fare target are removed before the train/test split. All feature imputation, encoding, and scaling are still learned within the training-only pipeline.

In [ ]:
FARE_FEATURES = ["pclass", "sex", "age", "sibsp", "parch", "embarked"]
fare_data = titanic.dropna(subset=["fare"]).copy()
X_fare = fare_data[FARE_FEATURES].copy()
y_fare = pd.to_numeric(fare_data["fare"], errors="raise")

X_fare_train, X_fare_test, y_fare_train, y_fare_test = train_test_split(
    X_fare, y_fare, test_size=0.20, random_state=RANDOM_STATE
)
fare_preprocessor = ColumnTransformer(
    [
        (
            "numeric",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            ["pclass", "age", "sibsp", "parch"],
        ),
        (
            "categorical",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            ["sex", "embarked"],
        ),
    ],
    remainder="drop",
)
fare_pipeline = Pipeline(
    [
        ("preprocessor", fare_preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300, random_state=RANDOM_STATE, n_jobs=1
            ),
        ),
    ]
)
fare_pipeline.fit(X_fare_train, y_fare_train)
fare_prediction = fare_pipeline.predict(X_fare_test)
fare_metrics = {
    "MAE": mean_absolute_error(y_fare_test, fare_prediction),
    "RMSE": np.sqrt(mean_squared_error(y_fare_test, fare_prediction)),
    "R-squared": fare_pipeline.score(X_fare_test, y_fare_test),
}
display(pd.Series(fare_metrics).round(3))


## 6. Serialize, reload, and predict

Save the tuned classification pipeline and fare-regression pipeline together with their preprocessing steps. The artifacts are written under `analytics/models/` whether the notebook is launched from the project root or from `analytics/`.

In [ ]:
current_directory = Path.cwd()
analytics_directory = (
    current_directory
    if current_directory.name.lower() == "analytics"
    else current_directory / "analytics"
)
model_directory = analytics_directory / "models"
model_directory.mkdir(parents=True, exist_ok=True)

classification_artifact = model_directory / "titanic_survival_pipeline.joblib"
regression_artifact = model_directory / "titanic_fare_regression_pipeline.joblib"
joblib.dump(best_pipeline, classification_artifact)
joblib.dump(fare_pipeline, regression_artifact)
print(f"Saved classification pipeline: {classification_artifact.resolve()}")
print(f"Saved fare-regression pipeline: {regression_artifact.resolve()}")


In [ ]:
loaded_classifier = joblib.load(classification_artifact)
loaded_regressor = joblib.load(regression_artifact)

classification_example = X_test.head(1)
fare_example = X_fare_test.head(1)
print("Reloaded classification prediction:", loaded_classifier.predict(classification_example)[0])
print("Reloaded fare prediction:", round(float(loaded_regressor.predict(fare_example)[0]), 2))


## 7. Interpretation and limitations

- Compare the tuned model with the simple baseline and the three initial classifiers using the held-out metrics, especially ROC-AUC and the precision/recall trade-off.
- The cross-validation score guided model selection; the OOB score is an internal bootstrap diagnostic; the test score is the final holdout estimate. These answer different questions.
- Passenger records are observational and historically specific. The metrics do not establish causal relationships, guarantee performance on another population, or remove biases in the source data.
- For a production decision, validate on representative data, document the cost of false positives/negatives, and monitor model behavior over time.